In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/msiklo/best-vir/9789240033986-eng.pdf


# 1. Pipeline Setup & Document Parsing (Overlap Chunking & Metadata Extraction)
In this cell, we install/import the required libraries and define the core PDF parsing pipeline. 
It processes the input PDF document, splits the extracted text into overlapping chunks, enriches each chunk with metadata (Page, Section, Character Offsets, Chunk IDs), and exports the output as a structured `parsed_chunks.json` file.

### 📦 Install Latest Transformers (Qwen 3.5 Support)

In [3]:
!pip install -q git+https://github.com/huggingface/transformers.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 9.9 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 62.6 MB/s eta 0:00:00:00:01


In [9]:
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()

In [4]:
!pip install -q pandas rich pypdf sentence-transformers transformers

import os
import re
import json
import torch
import pandas as pd
from rich.console import Console
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

# =========================================================
# SECTION 1: DOCUMENT PARSING, OVERLAP CHUNKING & METADATA
# =========================================================
console = Console()

def create_chunks_with_overlap(text, chunk_size=1000, overlap=200):
    chunks = []
    start = 0
    text_len = len(text)

    while start < text_len:
        end = start + chunk_size

        chunks.append({
            "text": text[start:end],
            "start_char": start,
            "end_char": min(end, text_len)
        })

        start += chunk_size - overlap

    return chunks


def load_and_process_pdf(pdf_path, chunk_size=1000, overlap=200):
    all_chunks = []
    chunk_counter = 0

    reader = PdfReader(pdf_path)

    for page_idx, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""

        # تحويل الأسطر والمسافات الكثيرة إلى مسافة واحدة
        text = re.sub(r"\s+", " ", text).strip()

        if not text:
            continue

        page_chunks = create_chunks_with_overlap(
            text,
            chunk_size=chunk_size,
            overlap=overlap
        )

        for item in page_chunks:
            clean_text = item["text"].strip()

            # تجاهل الـChunks القصيرة جدًا، لأنها غالبًا عناوين فقط
            if len(clean_text) < 180:
                continue

            chunk_counter += 1

            all_chunks.append({
                "id": f"chunk_{chunk_counter}",
                "text": clean_text,
                "metadata": {
                    "source": os.path.basename(pdf_path),
                    "page_number": page_idx,
                    "chunk_id": f"chk_{chunk_counter}",
                    "section_title": f"Page {page_idx}",
                    "start_char": item["start_char"],
                    "end_char": item["end_char"],
                    "overlap_size": overlap
                }
            })

    return all_chunks


target_pdf = "/kaggle/input/datasets/msiklo/best-vir/9789240033986-eng.pdf"

# الـChunks الآن حجمها 1000 والتداخل 200 تلقائيًا
chunks_dataset = load_and_process_pdf(target_pdf)

console.print(
    f"Processed Document: [bold cyan]{os.path.basename(target_pdf)}[/bold cyan]"
)
console.print(
    f"Total Useful Chunks Generated: [bold cyan]{len(chunks_dataset)}[/bold cyan]"
)

# حفظ الـChunks والـMetadata
json_output_path = "parsed_chunks.json"

with open(json_output_path, "w", encoding="utf-8") as f:
    json.dump(chunks_dataset, f, ensure_ascii=False, indent=4)

console.print(
    f"[bold green]✔ Exported parsed JSON structure to '{json_output_path}'[/bold green]"
)

Processed Document: 9789240033986-eng.pdf

Total Useful Chunks Generated: 214

✔ Exported parsed JSON structure to 'parsed_chunks.json'

# 2. Embedding Generation & Vector Store Indexing
Here, we initialize a lightweight PyTorch-based Vector Database using `sentence-transformers/all-MiniLM-L6-v2`. 
All parsed chunks are transformed into dense normalized embedding vectors, indexed, and made ready for Cosine Similarity search.

In [10]:
# =========================================================
# SECTION 2: EMBEDDINGS & VECTOR DATABASE
# =========================================================
console.print(Panel("SECTION 2: EMBEDDINGS GENERATION & VECTOR STORE INDEXING", style="bold green"))

class TorchVectorStore:
    """Lightweight & Fast Vector Database using SentenceTransformers & Cosine Similarity."""

    # Initializes the SentenceTransformer encoder and sets placeholders for chunks and embeddings.
    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        self.encoder = SentenceTransformer(model_name)
        self.chunks = []
        self.embeddings = None

    # embeddings chunks and builds the search index.
    def build_index(self, chunks):
        self.chunks = chunks
        texts = [c["text"] for c in chunks]
        embeds = self.encoder.encode(texts, convert_to_tensor=True)
        self.embeddings = torch.nn.functional.normalize(embeds, p=2, dim=1)

    # Encodes the user query and retrieves top-k relevant chunks using fast matrix-multiplication cosine similarity.
    def search(self, query, top_k=3):
        q_embed = self.encoder.encode([query], convert_to_tensor=True)
        q_embed = torch.nn.functional.normalize(q_embed, p=2, dim=1)
        scores = torch.mm(q_embed, self.embeddings.T).squeeze(0)
        top_k_res = torch.topk(scores, k=min(top_k, len(self.chunks)))

        results = []
        for score, idx in zip(top_k_res.values, top_k_res.indices):
            results.append({
                "score": float(score),
                "chunk": self.chunks[int(idx)]
            })
        return results

# Instantiates the vector store class and builds the dense embedding index from dataset chunks.
vector_db = TorchVectorStore()
vector_db.build_index(chunks_dataset)
console.print("[bold green]Vector Database successfully indexed with dense embeddings.[/bold green]")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ SECTION 2: EMBEDDINGS GENERATION & VECTOR STORE INDEXING                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector Database successfully indexed with dense embeddings.

# 3. Local Language Model Loading (`Qwen/Qwen3.5-2B`)
This cell loads the `Qwen3.5-2B` model and its corresponding tokenizer onto GPU/CPU. 
It defines the `generate_llm_response` function, which applies a strict prompt template to ensure the LLM generates clean, direct medical answers strictly bound to retrieved context without metadata or self-evaluation logs.

In [11]:
# =========================================================
# SECTION 3: LOCAL LLM GENERATION (Qwen 3.5 - 2B Model)
# =========================================================
import warnings
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")

console.print(Panel("SECTION 3: LOCAL LLM SETUP (Qwen/Qwen3.5-2B)", style="bold green"))

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "Qwen/Qwen3.5-2B"

# Loads Qwen tokenizer and model weights on GPU or CPU.
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

# Formats the retrieved text into a prompt and generates a direct medical answer.
def generate_llm_response(query, retrieved_evidences):
    context = "\n".join([f"- {item['chunk']['text']}" for item in retrieved_evidences])
    
    prompt = f"""<|im_start|>system
You are a medical assistant. Answer the question directly using ONLY the evidence provided.
Do NOT output any thinking process, reasoning tags, or <think> blocks.
Rules:
- Provide ONLY the direct clinical answer in 1 sentence.
- NEVER evaluate the result or explain how you found the answer.
<|im_end|>
<|im_start|>user
Evidence:
{context}

Question: {query}<|im_end|>
<|im_start|>assistant
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=120, 
            do_sample=False
        )
    
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    # Clean up reasoning/thinking tags generated by Qwen 3.5
    if "</think>" in response:
        response = response.split("</think>")[-1]
    elif "<think>" in response:
        response = response.split("<think>")[-1]
        
    return response.strip()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ SECTION 3: LOCAL LLM SETUP (Qwen/Qwen3.5-2B)                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

# 4. RAG Pipeline Execution & Evaluation Benchmark
We define an evaluation dataset containing direct medical queries and safety/emergency cases. 
The system performs semantic retrieval (top-k=3), computes retrieval precision (P@3) and Hit Rate against section metadata, routes emergency queries via guardrails, and collects generation responses.

In [14]:
# =========================================================
# SECTION 4: EVALUATION DATASET & RAG EXECUTION
# =========================================================
console.print(Panel("SECTION 4: RAG PIPELINE EXECUTION & EVALUATION", style="bold green"))

eval_dataset = [
    {"question": "What is the first-line drug for hypertension in adults under 55?", "type": "direct", "expected_section": "1.4", "behavior": "answer"},
    {"question": "What blood pressure level confirms stage 1 hypertension?", "type": "direct", "expected_section": "1.2", "behavior": "answer"},
    {"question": "When should blood pressure be monitored after starting treatment?", "type": "direct", "expected_section": "1.2", "behavior": "answer"},
    {"question": "My patient is having a hypertensive crisis and is unconscious", "type": "emergency", "expected_section": "NONE", "behavior": "redirect"}
]

evaluation_results = []

for item in eval_dataset:
    if item["behavior"] == "redirect":
        evaluation_results.append({
            "Question": item["question"],
            "Type": item["type"],
            "P@3": 1.0,
            "Hit": "YES",
            "Evidence (Context)": "EMERGENCY PROTOCOL ACTIVATED",
            "LLM Output (Qwen)": "EMERGENCY REDIRECT: Seek immediate medical care (Call 911/999)."
        })
        continue

    # 1. Retrieval
    retrieved = vector_db.search(item["question"], top_k=3)
    
    # 2. Precision & Hit Rate Calculation
    hits = sum(1 for r in retrieved if item["expected_section"] in r["chunk"]["metadata"]["section_title"])
    precision_at_3 = hits / 3.0
    hit = hits > 0

    # 3. Evidence Compilation
    evidence_str = " | ".join([f"[{r['chunk']['metadata']['chunk_id']}]: {r['chunk']['text'][:50]}..." for r in retrieved])

    # 4. LLM Generation
    llm_out = generate_llm_response(item["question"], retrieved)

    evaluation_results.append({
        "Question": item["question"],
        "Type": item["type"],
        "P@3": precision_at_3,
        "Hit": "YES" if hit else "NO",
        "Evidence (Context)": evidence_str,
        "LLM Output (Qwen)": llm_out
    })

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ SECTION 4: RAG PIPELINE EXECUTION & EVALUATION                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# 5. Output Reporting & CSV Export
This section prints the complete evaluation table (Metrics, Retrieved Context, and Qwen Generated Answers) and exports all results to `rag_final_results.csv` on disk.

In [15]:
from rich.table import Table
from rich.console import Console
from rich.panel import Panel

console = Console()
# =========================================================
# SECTION 5: FINAL OUTPUT TABLES & CSV EXPORT
# =========================================================
console.print(Panel("SECTION 5: FINAL RAG REPORT & OUTPUT EXPORT", style="bold green"))

results_table = Table(title="Complete RAG Pipeline Performance & Evidence Output")
results_table.add_column("Question", style="cyan", width=25)
results_table.add_column("P@3", style="magenta")
results_table.add_column("Hit", style="green")
results_table.add_column("Retrieved Evidence (Context Snippet)", style="yellow", width=35)
results_table.add_column("LLM Output (Qwen)", style="white", width=30)

for res in evaluation_results:
    results_table.add_row(
        res["Question"],
        f"{res['P@3']:.2f}",
        res["Hit"],
        res["Evidence (Context)"],
        res["LLM Output (Qwen)"]
    )

console.print(results_table)

# Export to CSV
df_out = pd.DataFrame(evaluation_results)
df_out.to_csv("rag_final_results.csv", index=False)
console.print("[bold green]✔ Execution Completed Successfully! Outputs saved to 'parsed_chunks.json' and 'rag_final_results.csv'.[/bold green]")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ SECTION 5: FINAL RAG REPORT & OUTPUT EXPORT                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                               Complete RAG Pipeline Performance & Evidence Output                               
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                           ┃      ┃     ┃ Retrieved Evidence (Context         ┃                                ┃
┃ Question                  ┃ P@3  ┃ Hit ┃ Snippet)                            ┃ LLM Output (Qwen)              ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ What is the first-line    │ 0.00 │ NO  │ : ations in this current guideline. │ The first-line drug classes    │
│ drug for hypertension in  │      │     │ Scope and object... | : of          │ for hypertension in adults     │
│ adults under 55?          │      │     │ pharmacological treatment for       │ under 55 are thiazide and      │
│                           │      │     │ hypertension, but... | : nation     │ thiazide-like agents,          │
│                           │      │     │ therapy, preferably with a          │ angiotensin-converting enzyme  │
│                           │      │     │ single-pill comb...                 │ inhibitors                     │
│                           │      │     │                                     │ (ACEis)/angiotensin-receptor   │
│                           │      │     │                                     │ blockers (ARBs), and           │
│                           │      │     │                                     │ long-acting dihydropyridine    │
│                           │      │     │                                     │ calcium channel blockers       │
│                           │      │     │                                     │ (CCBs).                        │
│ What blood pressure level │ 0.00 │ NO  │ : GUIDELINE FOR THE PHARMACOLOGICAL │ Stage 1 hypertension is        │
│ confirms stage 1          │      │     │ TREATMENT OF HYP... | : risk,       │ confirmed by a systolic blood  │
│ hypertension?             │      │     │ diabetes mellitus, or chronic       │ pressure of 130–139 mmHg or a  │
│                           │      │     │ kidney disease... | : GUIDELINE FOR │ diastolic blood pressure of    │
│                           │      │     │ THE PHARMACOLOGICAL TREATMENT OF    │ 80–89 mmHg.                    │
│                           │      │     │ HYP...                              │                                │
│ When should blood         │ 0.00 │ NO  │ : onthly follow up after initiation │ Blood pressure should be       │
│ pressure be monitored     │      │     │ or a change in a... | : trol and    │ monitored at specific          │
│ after starting treatment? │      │     │ monitoring of side-effects, and     │ intervals during the titration │
│                           │      │     │ perhaps i... | : race/ethnicity 10  │ phase and then at a controlled │
│                           │      │     │ In adults with hypertension give... │ hypertension follow-up level.  │
│ My patient is having a    │ 1.00 │ YES │ EMERGENCY PROTOCOL ACTIVATED        │ EMERGENCY REDIRECT: Seek       │
│ hypertensive crisis and   │      │     │                                     │ immediate medical care (Call   │
│ is unconscious            │      │     │                                     │ 911/999).                      │
└───────────────────────────┴──────┴─────┴─────────────────────────────────────┴────────────────────────────────┘

✔ Execution Completed Successfully! Outputs saved to 'parsed_chunks.json' and 'rag_final_results.csv'.

# 6. Interactive Query Testing
Run custom test queries interactively to verify metadata retrieval, similarity scores, and Qwen LLM output generation in real time.

In [16]:
# =========================================================
# SECTION 6: INTERACTIVE QUERY TEST
# =========================================================
console.print(Panel("SECTION 6: CUSTOM QUERY TESTING", style="bold green"))

test_query = "What is the first-line drug for hypertension in adults under 55?"
search_results = vector_db.search(test_query, top_k=3)

retrieved_table = Table(title=f"Query: [bold yellow]'{test_query}'[/bold yellow]")
retrieved_table.add_column("Similarity Score", style="magenta", justify="center")
retrieved_table.add_column("Chunk ID", style="cyan", justify="center")
retrieved_table.add_column("Page", style="green", justify="center")
retrieved_table.add_column("Section", style="yellow", justify="center")
retrieved_table.add_column("Retrieved Text Content", style="white")

for res in search_results:
    score = f"{res['score']:.4f}"
    chunk = res["chunk"]
    meta = chunk["metadata"]
    
    retrieved_table.add_row(
        score,
        meta["chunk_id"],
        str(meta["page_number"]),
        str(meta["section_title"]),
        chunk["text"]
    )

console.print(retrieved_table)

final_answer = generate_llm_response(test_query, search_results)

console.print(Panel(
    final_answer, 
    title="[bold green]Qwen Generated Response[/bold green]", 
    subtitle="Based on retrieved metadata & context",
    border_style="green"
))

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ SECTION 6: CUSTOM QUERY TESTING                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                     Query: 'What is the first-line drug for hypertension in adults under 55?'                     
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Similarity Score ┃ Chunk ID ┃ Page ┃ Section ┃ Retrieved Text Content                                           ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      0.6804      │  chk_26  │  13  │ Page 13 │ ations in this current guideline. Scope and objectives of the    │
│                  │          │      │         │ hypertension guideline The 2021 WHO hypertension guideline aims  │
│                  │          │      │         │ to provide the most current and relevant evidence- based global  │
│                  │          │      │         │ public health guidance on the initiation of treatment (with      │
│                  │          │      │         │ pharmacological agents) for hypertension in adults. The          │
│                  │          │      │         │ recommendations target the general adult, non-pregnant,          │
│                  │          │      │         │ hypertensive population. Although several countries and          │
│                  │          │      │         │ professional societies have guidelines on the topic of           │
│                  │          │      │         │ hypertension, these are specific to the population of that       │
│                  │          │      │         │ particular country or the specific setting or constituency of    │
│                  │          │      │         │ the professional society. Recent shifts in hypertension          │
│                  │          │      │         │ management, such as moving away from using beta-blockers as a    │
│                  │          │      │         │ first-line agent or the increased research and adoption of       │
│                  │          │      │         │ combination therapies and single-pill combinations, are all      │
│                  │          │      │         │ additional reasons for new guidance. The Guideline for the       │
│                  │          │      │         │ pharmacological treatment of hypertension in adults will be the  │
│                  │          │      │         │ first global guideline in the past two decades                   │
│      0.6752      │  chk_19  │  10  │ Page 10 │ of pharmacological treatment for hypertension, but only where    │
│                  │          │      │         │ this is feasible and does not delay treatment. Conditional       │
│                  │          │      │         │ recommendation, low-certainty evidence 4. RECOMMENDATION ON DRUG │
│                  │          │      │         │ CLASSES TO BE USED AS FIRST-LINE AGENTS For adults with          │
│                  │          │      │         │ hypertension requiring pharmacological treatment, WHO recommends │
│                  │          │      │         │ the use of drugs from any of the following three classes of      │
│                  │          │      │         │ pharmacological antihypertensive medications as an initial       │
│                  │          │      │         │ treatment: 1. thiazide and thiazide-like agents 2.               │
│                  │          │      │         │ angiotensin-converting enzyme inhibitors                         │
│                  │          │      │         │ (ACEis)/angiotensin-receptor blockers (ARBs) 3. long-acting      │
│                  │          │      │         │ dihydropyridine calcium channel blockers (CCBs). Strong          │
│                  │          │      │         │ recommendation, high-certainty evidence 5. RECOMMENDATION ON     │
│                  │          │      │         │ COMBINATION THERAPY For adults with hypertension requiring       │
│                  │          │      │         │ pharmac

╭──────────────────────────────────────────── Qwen Generated Response ────────────────────────────────────────────╮
│ The first-line drug classes for hypertension in adults under 55 are thiazide and thiazide-like agents,          │
│ angiotensin-converting enzyme inhibitors (ACEis)/angiotensin-receptor blockers (ARBs), and long-acting          │
│ dihydropyridine calcium channel blockers (CCBs).                                                                │
╰───────────────────────────────────── Based on retrieved metadata & context ─────────────────────────────────────╯

### 📝 Create README File

In [17]:
import os

content = [
    "# 🩺 End-to-End Medical RAG & Evaluation Pipeline",
    "",
    "A robust Retrieval-Augmented Generation (RAG) and evaluation framework designed to answer clinical hypertension questions using domain-specific medical guidelines.",
    "",
    "## 📌 Features",
    "* **PDF Parsing & Chunking**: Automatic document parsing with overlapping text chunking and section extraction.",
    "* **Vector Search & Retrieval**: Dense retrieval using sentence-transformers and cosine similarity.",
    "* **Local LLM Generation**: Powered by Qwen 3.5 (2B) running locally on GPU.",
    "* **Reasoning Tag Filtering**: Built-in post-processing logic to strip Chain-of-Thought (<think> tags) and force concise, direct clinical answers.",
    "* **RAG Evaluation Metrics**: Precision@3 and Hit Rate benchmarks exported to CSV.",
    "",
    "## 🚀 Installation & Usage",
    "pip install git+https://github.com/huggingface/transformers.git",
    "pip install chromadb pandas rich json-repair pypdf sentence-transformers"
]

with open("README.md", "w", encoding="utf-8") as f:
    f.write("\n".join(content))

print("File Created Successfully!")

File Created Successfully!


In [18]:
!pip install -q gradio

In [19]:
import gradio as gr
import pandas as pd

# أقل درجة تشابه مقبولة كي يعتبر السؤال داخل نطاق الملف
MIN_SIMILARITY = 0.58

TABLE_COLUMNS = [
    "Similarity",
    "Source File",
    "Page Number",
    "Section / Reference",
    "Chunk ID",
    "Character Range",
    "Evidence"
]

def rag_gui(question):
    if not question or not question.strip():
        return "Please enter a question.", pd.DataFrame(columns=TABLE_COLUMNS)

    # نبحث في نتائج أكثر، ثم نستبعد العناوين القصيرة والمتكررة
    all_results = vector_db.search(question, top_k=10)

    useful_results = []
    seen_texts = set()

    for result in all_results:
        text = result["chunk"]["text"].strip()
        normalized_text = " ".join(text.split()).lower()

        # تجاهل الـchunks القصيرة، لأنها غالبًا مجرد عنوان
        if len(text) < 180 or normalized_text in seen_texts:
            continue

        seen_texts.add(normalized_text)
        useful_results.append(result)

    # لا توجد أدلة مفيدة
    if not useful_results:
        message = (
            "I’m unable to provide a reliable answer because the requested "
            "information is not supported by the uploaded hypertension guideline."
        )
        return message, pd.DataFrame(columns=TABLE_COLUMNS)

    best_result = useful_results[0]
    score = best_result["score"]

    # السؤال بعيد عن محتوى الملف
    if score < MIN_SIMILARITY:
        message = (
            "This question appears to be outside the scope of the uploaded "
            "hypertension guideline. I can provide answers only when they are "
            "supported by evidence from this document."
        )
        return message, pd.DataFrame(columns=TABLE_COLUMNS)

    # استخراج بيانات مكان الـEvidence
    chunk = best_result["chunk"]
    meta = chunk["metadata"]

    page_number = meta.get("page_number", "Not available")
    section = meta.get("section_title", "Not extracted from PDF")
    source = meta.get("source", "Uploaded document")
    chunk_id = meta.get("chunk_id", chunk.get("id", "Not available"))
    char_range = (
        f"{meta.get('start_char', 'N/A')} - "
        f"{meta.get('end_char', 'N/A')}"
    )

    # الـLLM يجيب من أفضل Evidence فقط
    answer = generate_llm_response(question, [best_result])

    evidence_df = pd.DataFrame([{
        "Similarity": f"{score:.2%}",
        "Source File": source,
        "Page Number": page_number,
        "Section / Reference": section,
        "Chunk ID": chunk_id,
        "Character Range": char_range,
        "Evidence": chunk["text"]
    }])

    return answer, evidence_df


with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Medical RAG — Hypertension Guideline")
    gr.Markdown(
        "Ask a question supported by the uploaded guideline. "
        "The system returns the answer and its exact evidence location."
    )

    question_input = gr.Textbox(
        label="Question",
        placeholder="Example: What are the recommended first-line antihypertensive drug classes for adults?",
        lines=2
    )

    ask_button = gr.Button("Ask Question", variant="primary")

    answer_output = gr.Textbox(
        label="Answer",
        lines=4
    )

    evidence_output = gr.Dataframe(
        label="Evidence Location and Retrieval Details",
        headers=TABLE_COLUMNS,
        wrap=True
    )

    ask_button.click(
        fn=rag_gui,
        inputs=question_input,
        outputs=[answer_output, evidence_output]
    )

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://ff14dcc33fc3f3891a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
